RNN - Erro dos pesos computados e usado somente durante a iteração

In [10]:
import numpy as np
from numpy import linalg as LA
import pandas as pd
import ipynbname
import optuna
from optuna.samplers import RandomSampler
from optuna.samplers import TPESampler
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_pareto_front
from optuna.importance import get_param_importances
from optuna.exceptions import TrialPruned
import matplotlib as mpl
#from Testing.RTLO import *
from Functions.RLS import *
from Functions.Utils_RTLO import *
from Functions.Graphs import *
from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.metrics import mean_squared_error as MSE
FileName = ipynbname.name()

df = pd.read_csv(r'Dataset\Bearing1_1.csv')
df = pd.read_excel(r'Dataset\Bearing1_1.xlsx')
sig = df['PC1'].values

    
def SelSampler(mode='auto'):
    '''mode: auto, random,  tpe'''
    if mode == 'auto':
        sampler = None
    elif mode == 'tpe':
        sampler = optuna.samplers.TPESampler(multivariate=True, constant_linker=True,group=True,n_startup_trials=2000)
    elif mode == 'random':
        sampler=RandomSampler()
    return sampler


In [2]:
class RTLO:
    def __init__(self, nI,nR,nO,ηS=[0.1,0.1,0.1], τ=10,mode='past'):
        np.random.seed(42)
        self.k = 1
        self.j = nI-1
        self.t = np.array([])
        self.ref = None
        self.act = 'tanh'
        self.flw = mode
        self.n = -1
        self.nI, self.nR, self.nO = nI, nR, nO

        self.ηS = np.array(ηS)
        self.τ = τ
        self.ρ = 0.008

        self.rls = RLS_LogarithmicRegressor(0.9,1e7)

        self.xPi = np.zeros(nI)
        self.hP, self.hU, self.hL = [0.1*np.ones(nR) for i in range(3)]

        self.pS = np.zeros((self.nR, self.nR))
        self.qS = np.zeros((self.nR, self.nI))

        self.ΔOS = np.zeros((nO, nR))
        self.ΔRS = np.zeros((nR, nR))
        self.ΔIS = np.zeros((nR, nI))
        
        self.wI = XavierUniform([nR, nI],sd=42)
        self.wR = XavierUniform([nR, nR],sd=41)
        self.wO = XavierUniform([nO, nR],sd=40)
        self.BS = XavierUniform([nR, nO],sd=39)
        #self.BS = np.random.randn(nR, nO)/nO**0.5
        
        self.yP, self.yR, self.yL, self.yU = [np.array([]) for i in range(4)]
        self.yP_hist = np.zeros(self.nI)

        self.εY, self.εM, self.εR, self.εE, self.eP, self.eR, self.ΣW = [0 for i in range(7)]
        self.εM_hist,self.εR_hist, self.eR_hist, self.eP_hist = [np.array([]) for i in range(4)]

        self.wR_hist = []
        self.wI_hist = []
        self.wO_hist = []

        self.rR = 1e-9
        self.rP = 1e-10
        self.rL = 1e-11
        self.rU = 1e-12
        self.rRsum = 0

        self.rulR, self.rulP, self.rulL, self.rulU = [np.array([]) for i in range(4)]


    def PredSingle(self,x):

        u = np.dot(self.wR, self.hS) + np.dot(self.wI, x)
        h = self.hS + (-self.hS + Activation(u,self.act))/self.τ
        y = np.dot(self.wO, h)

        return y

    def fit(self,xP,yR,start=0,store=False,show=False):
        if self.flw != 'past': self.n = 0
        
        η1,η2,η3 = self.ηS      

        uS = self.wR @ self.hP + self.wI @ xP
        hP = self.hP*(1-1/self.τ) +Activation(uS,self.act)/self.τ
        yP = self.wO @ hP
        eS = yR-yP

        self.pS = np.outer(dActivation(uS,self.act),self.hP)/self.τ + (1-1/self.τ)*self.pS
        self.qS = np.outer(dActivation(uS,self.act),self.xPi)/self.τ + (1-1/self.τ)*self.qS

        δOS = η1*np.outer(eS,hP)
        δRS = η2*np.outer((self.BS@eS),np.ones(self.nR))*self.pS
        δIS = η3*np.outer(np.dot(self.BS, eS),np.ones(self.nI))*self.qS

        self.wI = self.wI + δIS
        self.wR = self.wR + δRS
        self.wO = self.wO + δOS

        self.wR_hist.append(self.wR.flatten())
        self.wI_hist.append(self.wI.flatten())
        self.wO_hist.append(self.wO.flatten())

        self.hP = hP
        self.xPi = xP

        #if self.k<start:
            #self.rulL = np.append(self.rulL,self.rP)
            #self.rulU = np.append(self.rulU,self.rP)

        if self.k>=start:
            #self.UpdateRLS(store)
            W = self.k**2
            ΣW = self.ΣW + W
            ΔY = np.abs((yR-yP)/(yR+1e-9))
            ΔM = np.linalg.norm(ΔY,ord=2)
            ΔR = np.abs((self.rR-self.rP)/(self.rR+1e-9))
            ΔE = np.abs((self.eP-self.eR)/(self.eR+1e-9))

            self.εY = ((self.εY*self.ΣW) + (W*ΔY[self.n]))/ΣW
            self.εM = ((self.εM*self.ΣW) + (W*ΔM))/ΣW
            self.εR = ((self.εR*self.ΣW) + (W*ΔR))/ΣW
            self.εE = ((self.εE*self.ΣW) + (W*ΔE))/ΣW
            self.ΣW = ΣW

        if store:
            self.yR = np.append(self.yR,yR[self.n])
            if self.k>=start:
                self.εM_hist = np.append(self.εM_hist,self.εM)
                self.εR_hist = np.append(self.εR_hist,self.εR)

        self.k = self.k+1
        self.t = np.append(self.t,self.k + self.nI)
        self.yP_hist = np.delete(np.append(self.yP_hist,yP[0]),0)
        #self.ηS = self.ηS/(1 + self.decay*self.k)


    '''def PredRulIntr(self, x,lim=0.2,maxRul=110,store=False,show=False):
        #print('yH:',self.yP_hist)
        #print('eS: ',self.eS)

        for i,y in enumerate(self.yP_hist):
            if y != 0:
                self.eS2[i] = self.rls.predict(np.abs(y))
        #print('eS2:',self.eS2)

        xP,xL,xU =x.copy(), (x-(self.ρ*self.eS2)).copy(),(x+(self.ρ*self.eS2)).copy()    

        #xP,xL,xU =x.copy(), x.copy()*0.999, x.copy()*1.00

        predict = True
        PredRuls = [True for i in range(3)]
        PredVals, Ruls = [0 for i in range(3)], [0 for i in range(3)]
        wR,wI,wO = self.wR,self.wI,self.wO

        wRp, wRn = np.maximum(0, self.wR), np.abs(np.minimum(0, self.wR))
        wIp, wIn = np.maximum(0, self.wI), np.abs(np.minimum(0, self.wI))
        wOp, wOn = np.maximum(0, self.wO), np.abs(np.minimum(0, self.wO))
        #hU,hL = np.maximum(0, self.hS), np.minimum(0, self.hS)
        hL,hP,hU = [self.hP.copy() for i in range(3)]

        while predict:
            uP = wR@hP + wI@xP
            uL = (wRp @ hL - wRn @ hU) + (wIp @ xL - wIn @ xU)
            uU = (wRp @ hU - wRn @ hL) + (wIp @ xU - wIn @ xL)

            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            hL = hL*(1-1/self.τ) + Activation(uL,self.act)/self.τ
            hU = hU*(1-1/self.τ) + Activation(uU,self.act)/self.τ
    
            yP = (wO@hP)
            yL = (wOp @ hL - wOn @ hU)
            yU = (wOp @ hU - wOn @ hL)

            #if show: print(yP)
            yP = yP[0]
            yL = yL[0]
            yU = yU[0]

            xP = np.delete(np.append(xP,yP),0)
            xL = np.delete(np.append(xL,yL),0)
            xU = np.delete(np.append(xU,yU),0)
            PredVals = [yL,yP,yU]

            if Ruls[0] == 0:
                self.yL = np.append(self.yL,yL)
                self.yP = np.append(self.yP,yP)
                self.yU = np.append(self.yU,yU)
            
            CheckPred,CheckLim=0,0
            for i in range(3):
                if PredRuls[i]: 
                    Ruls[i] = Ruls[i]+1
                    if Ruls[i] >= maxRul:
                        CheckLim = CheckLim + 1
                        Ruls[i] = maxRul
                        PredRuls[i] = False
                if PredVals[i] < lim: PredRuls[i] = False
                if not PredRuls[i]: CheckPred = CheckPred + 1
            if CheckPred == 3:break
            if CheckLim == 3:break
        
        self.rR=self.ref-self.k
        self.rL,self.rP,self.rU = Ruls

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulL = np.append(self.rulL,self.rL)
            self.rulP = np.append(self.rulP,self.rP)
            self.rulU = np.append(self.rulU,self.rU)'''
    
    def PredRulIntr2(self, x,lim=0.2,maxRul=110,store=False,show=False):
        if self.flw != 'past': self.n = 0
        xP,xL,xU =x.copy(), x.copy(), x.copy()
        k = 1
        predict = True
        PredRuls = [True for i in range(3)]
        PredVals, Ruls = [0 for i in range(3)], [0 for i in range(3)]
        wR,wI,wO = self.wR,self.wI,self.wO
        hP,hU,hL = [self.hP.copy() for i in range(3)]

        wRU, wRL = np.maximum((1+self.ρ)*wR, wR/(1+self.ρ)), np.minimum((1+self.ρ)*wR, wR/(1+self.ρ))
        wIU, wIL = np.maximum((1+self.ρ)*wI, wI/(1+self.ρ)), np.minimum((1+self.ρ)*wI, wI/(1+self.ρ))
        wOU, wOL = np.maximum((1+self.ρ)*wO, wO/(1+self.ρ)), np.minimum((1+self.ρ)*wO, wO/(1+self.ρ)) 
  
        while predict:

            uP = ( wR @ hP) + ( wI @ xP)
            uL = (wRL @ hL) + (wIL @ xL)
            uU = (wRU @ hU) + (wIU @ xU)
            uU, uL = np.maximum(uU,uL), np.minimum(uU,uL)
            
            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            hL = hL*(1-1/self.τ) + Activation(uL,self.act)/self.τ
            hU = hU*(1-1/self.τ) + Activation(uU,self.act)/self.τ
            hU, hL = np.maximum(hU,hL), np.minimum(hU,hL)

            yP = ( wO @ hP)
            yL = (wOL @ hL)
            yU = (wOU @ hU)
            yU, yL = np.maximum(yU, yL), np.minimum(yU, yL)

            if show:   
                print(k,yL,yP,yU)

            PredVals = ([yL[self.n],yP[self.n],yU[self.n]])
            yL,yP,yU = PredVals

            xP = np.delete(np.append(xP,yP),0)
            xL = np.delete(np.append(xL,yL),0)
            xU = np.delete(np.append(xU,yU),0)

            if Ruls[0] == 0:
                self.yL = np.append(self.yL,yL)
                self.yP = np.append(self.yP,yP)
                self.yU = np.append(self.yU,yU)
            
            CheckPred,CheckLim=0,0
            for i in range(3):
                if PredRuls[i]: 
                    Ruls[i] = Ruls[i]+1
                    if Ruls[i] >= maxRul:
                        CheckLim = CheckLim + 1
                        Ruls[i] = 1
                        PredRuls[i] = False
                if PredVals[i] < lim: PredRuls[i] = False
                if not PredRuls[i]: CheckPred = CheckPred + 1
            if CheckPred == 3:break
            if CheckLim == 3:break
            k = k+1

        if Ruls[1]< Ruls[0]: Ruls[1]=Ruls[0]   
        if Ruls[2]< Ruls[1]: Ruls[2]=Ruls[1]  +1      
        self.rR=self.ref-self.k
        self.rL,self.rP,self.rU = Ruls

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulL = np.append(self.rulL,self.rL)
            self.rulP = np.append(self.rulP,self.rP)
            self.rulU = np.append(self.rulU,self.rU)
    
    def PredRul(self, x,maxRul=100,lim=0.2,store=False):
        if self.flw != 'past': self.n = 0
        xP = x.copy()
        rulP=0
        predict = True
        hP = self.hP.copy()
        while predict:
            uP = (self.wR @ hP) + (self.wI @ xP)
            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            yP = (self.wO @ hP)[self.n]
            xP = np.delete(np.append(xP,yP),0)
            if store:
                if rulP==0:
                    self.yP = np.append(self.yP,yP)

            if predict: rulP = rulP+1
            if yP < lim: predict = False
            if rulP >= maxRul:
                rulP=1
                break
        self.rR=self.ref-self.k
        self.rP = rulP
        #print(self.rR, self.rP)
        #self.UpdateRLS()

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulP = np.append(self.rulP,self.rP)
            #rulL = int(self.rP-self.eP*self.ρ)
            #rulU = int(self.rP+self.eP*self.ρ)
            #if rulL<0: rulL = (rulP)
            #self.rulL = np.append(self.rulL,rulL)
            #self.rulU = np.append(self.rulU,rulU)
            
    def UpdateRLS(self,store=False):
        eP = int(np.abs(self.rls.predict(self.rP)))
        eR = np.abs(self.rP-self.rR)
        self.rls.update(np.abs(self.rP), eR)
        self.eP = eP
        self.eR = eR
        self.eR_hist = np.append(self.eR_hist,eR)
        self.eP_hist = np.append(self.eP_hist,eP)
        if store:
            rulL = int(self.rP-self.eP*self.ρ)
            rulU = int(self.rP+self.eP*self.ρ)
            if rulL<0: rulL = (self.rR)
            self.rulL = np.append(self.rulL,rulL)
            self.rulU = np.append(self.rulU,rulU)



#Optimize parameters for minimize error of degradation prediction

In [ ]:
rates = [1/(10**i) for i in range(1,8)][::-1]
def objective(trial):

    nI = trial.suggest_int('nI', 3, 3) 
    nR = trial.suggest_int('nR', 3, 3) 
    nO = trial.suggest_int('nO', 3, 3) 
    N1 = trial.suggest_categorical('N1', rates[:]) 
    N2 = trial.suggest_categorical('N2', rates[:]) 
    N3 = trial.suggest_categorical('N3', rates) 
    τ = trial.suggest_int('τ', 1, 40)    
    mode = trial.suggest_categorical('mode', ['past'])  
    cont=0
    if mode == 'past':
        if nO > nI:
            raise TrialPruned()
    X,Y = PrepareData(sig,nI,nO,mode)
    rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ,mode)
    rnn.ref = len(sig)-nI

    for i,_ in enumerate(X):
        rnn.PredRul(X[i],maxRul=len(sig)-nI,lim=0.75,store=True)
        rnn.fit(X[i],Y[i],store=True)

        if i == int((len(sig)-nI)/2):
            if np.mean(rnn.εM_hist)>1:
                raise TrialPruned()
            if np.mean(rnn.rulP)>np.mean(rnn.rulR)*1.25:
                raise TrialPruned()
            if np.mean(rnn.rulP)<np.mean(rnn.rulR)*0.25:
                raise TrialPruned()
        if i>=30:
            if rnn.rulP[i]==rnn.rulP[i-1]:
                cont = cont+1
            if cont> 20:
                raise TrialPruned()
            
    #return rnn.εY
    return rnn.εM + rnn.εR

#pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
pruner=optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    direction="minimize",
    sampler=SelSampler(mode='random'),
    pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)
study.optimize(objective, n_trials=20000)
params = list(study.best_params.values())
print('Erro:', study.best_value, 'parameters: ', params)

In [ ]:
rates = [1/(10**i) for i in range(1,9)][::-1]
def objective(trial):

    nI = trial.suggest_int('nI', 2, 40) 
    nR = trial.suggest_int('nR', 20, 50) 
    nO = trial.suggest_int('nO', 10, 30) 
    N1 = trial.suggest_categorical('N1', rates[2:]) 
    N2 = trial.suggest_categorical('N2', rates[2:]) 
    N3 = trial.suggest_categorical('N3', rates[:-1]) 
    τ = trial.suggest_int('τ', 10, 50)    
    mode = trial.suggest_categorical('mode', ['past','ahead'])  
    act = trial.suggest_categorical('act', ['relu','tanh'])  
    if mode == 'past':
        if nO > nI:
            raise TrialPruned()
    X,Y = PrepareData(sig,nI,nO,mode)
    rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ,mode)
    rnn.ref = len(sig)-nI

    for i,_ in enumerate(X):
        rnn.PredRul(X[i],maxRul=len(sig),lim=0.3,store=True)
        rnn.fit(X[i],Y[i],start=0,store=True)
        rnn.act=act

        if i == int((len(sig)-nI)/2):
            #if np.mean(rnn.εM_hist)>1:
            #    raise TrialPruned()
            if np.mean(rnn.rulP)>np.mean(rnn.rulR)*1.25:
                raise TrialPruned()
            if np.mean(rnn.rulP)<np.mean(rnn.rulR)*0.25:
                raise TrialPruned()
        '''if i>=30:
            if rnn.rulP[i]==rnn.rulP[i-1]:
                cont = cont+1
            if cont> 20:
                raise TrialPruned()'''
            
    return rnn.εM, rnn.εR

#pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
pruner=optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    directions=["minimize", "minimize"],
    sampler=SelSampler(mode='random'),
    #pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)
study.optimize(objective, n_trials=25000)

best_trials = study.best_trials

print(f"Encontrados {len(best_trials)} modelos na Fronteira de Pareto:")

for i, trial in enumerate(best_trials):
    print(f"Erro_M = {trial.values[0]:.6f}, Erro_R = {trial.values[1]:.6f}",
          f"Parâmetros: {trial.params}")
# Se você quiser apenas os parâmetros do PRIMEIRO modelo da fronteira para testar:
first_best_params = best_trials[0].params
params = list(trial.params.values())


Erro_M = 0.076862, Erro_R = 0.135507 Parâmetros: {'nI': 38, 'nR': 46, 'nO': 20, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.0001, 'τ': 14, 'mode': 'ahead', 'act': 'tanh'}


In [11]:
params =   {'nI': 38, 'nR': 46, 'nO': 20, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.0001, 'τ': 14, 'mode': 'ahead', 'act': 'tanh'}
params = list(params.values())

In [12]:
nI,nR,nO,N1,N2,N3,τ,mode,act= params
X,Y = PrepareData(sig,nI,nO,mode)
rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ,mode)
rnn.act=act
rnn.ref = len(sig)-nI
for i in range(len(X[:])):
    rnn.PredRulIntr2(x=X[i],maxRul=len(sig),lim=0.3,store=True,show=False)
    #rnn.PredRul(X[i],maxRul=len(sig),lim=0.75,store=True)

    
    rnn.fit(X[i],Y[i],store=True,start=0,show=False)
print(rnn.εM,rnn.εR)  
PlotPredErrorPLY(rnn,w=800,h=300)

0.07686189593108204 0.13550720899987642


In [27]:
i=int((len(sig)-nI)/2)
r_m = np.mean(rnn.rulR[:i])
r_mL = np.mean(rnn.rulR[:i])*0.35
r_mU = np.mean(rnn.rulR[:i])*1.25
p_m = (np.mean(rnn.rulP[:i]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)


εM_m = np.mean(rnn.εM_hist[:i])
εR_M = np.mean(rnn.εR_hist[:i])


print('εM_m:',εM_m)
print('εR_M:',εR_M)

lower: 31.849999999999998 mid: 91.0 upper: 113.75
pred: 27.34426229508197
εM_m: 1.1000916297717287
εR_M: 0.7947284274070086


In [ ]:
#params =  [14, 8, 14, 0.001, 0.01, 1e-06, 1e-07, 29]
nI,nR,nO,N1,N2,N3,τ,mode= params
ηS = [N1,N2,N3]
X,Y = PrepareData(sig,nI,nO,mode)
rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ,mode)
rnn.ref = len(sig)-nI

i=0

In [ ]:
#rnn.PredRul(x=X[i],store=True)
print('iter',i+5)
rnn.PredRulIntr2(x=X[i],maxRul=len(sig),lim=0.75,store=True,show=True)
rnn.fit(X[i],Y[i],store=True,start=0,show=False)
i=i+1